# 🤖 Proje #20: Yapay Zeka Asistanı Botu — Satır Satır Kod Açıklama Rehberi

Bu notebook dosyasında, **Flask** web çatısı ve **Google Gemini API** (`google-generativeai`) kullanılarak geliştirilen Yapay Zeka Asistanı uygulamamızın tüm kaynak kodlarını bölüm bölüm inceleyecek; hangi fonksiyonun ne işe yaradığını, neden kullanıldığını ve çalışma mantığını derinlemesine açıklayacağız.

## 📸 Uygulama Arayüzü

Aşağıda, tarayıcı üzerinden erişilen ve kullanıcıların interaktif olarak asistanla konuşabildiği web arayüzünün ekran görüntüsünü görebilirsiniz:

![Uygulama Arayüzü](public/projess.png)


## 1. Kütüphanelerin İçe Aktarılması (Imports)

Programın başında, web sunucusu, oturum yönetimi, sistem fonksiyonları ve yapay zeka API'si için gerekli kütüphaneleri yüklüyoruz.


In [ ]:
from flask import Flask, render_template, request, session, redirect, url_for
import os
import contextlib

# gRPC/Protobuf ve google-generativeai kütüphanesinin import aşamasında konsola yazdırdığı gereksiz uyarıları engellemek için
with open(os.devnull, 'w') as devnull, contextlib.redirect_stderr(devnull):
    import google.generativeai as genai


### 💡 Neden ve Ne İşe Yarar?
- **`Flask`**: Web sunucusunu ayağa kaldıran ana sınıftır.
- **`render_template`**: `templates/` klasöründeki HTML dosyalarını (şablonları) dinamik verilerle birleştirip tarayıcıya göndermeyi sağlar.
- **`request`**: Tarayıcıdan gönderilen form verilerine (`POST` parametreleri) veya URL parametrelerine (`GET`) erişmek için kullanılır.
- **`session`**: Kullanıcıya özgü verileri (örneğin sohbet geçmişi ve API anahtarı) tarayıcı çerezlerinde (cookie) güvenli ve şifreli saklamak için oturum yönetimi sağlar.
- **`redirect` & `url_for`**: Sayfalar arası yönlendirme ve dinamik URL oluşturma işlemleri için kullanılır.
- **`os`**: İşletim sistemi çevre değişkenlerinden (`os.environ`) API anahtarı okuyabilmek için içe aktarılır.
- **`contextlib.redirect_stderr`**: `google.generativeai` kütüphanesi içe aktarılırken arkada çalışan gRPC servisleri bazen konsola kırmızı loglar basabilir. `with` bloğu içerisinde standart hata çıktısını (`stderr`) çöpe (`devnull`) yönlendirerek terminalimizin temiz kalmasını sağlıyoruz.


## 2. Flask Başlatma ve Oturum Ayarı

Flask uygulamasını tanımlayıp, oturum şifrelemesi için gizli anahtar ataması yapıyoruz.


In [ ]:
app = Flask(__name__)
app.secret_key = "your_secret_key_here"  # Oturum şifreleme anahtarı


### 💡 Neden ve Ne İşe Yarar?
- **`Flask(__name__)`**: Flask'e şablonların ve statik dosyaların hangi dizinde aranacağını söyler.
- **`secret_key`**: Oturum verilerinin (session) istemci tarafından manipüle edilmesini engellemek için kriptografik bir imza oluşturur. Bu anahtar girilmezse Flask session kullanırken hata verir. Üretim ortamında buraya karmaşık ve gizli bir dize yazılmalıdır.


## 3. API Anahtarı Tanımlama ve Üretim Yapılandırması

API anahtarı kontrolü için statik değişkeni ve yapay zeka modelinin davranışlarını belirleyen parametreleri tanımlıyoruz.


In [ ]:
API_KEY_IN_CODE = "BURAYA APİ KEYİNİ GİR"

generation_config = {
    "temperature": 1,           # Yaratıcılık/rastgelelik düzeyi
    "top_p": 0.95,              # Kelime seçim olasılık kümülatifi
    "top_k": 40,                # Her adımda en olası kelime aday sayısı
    "max_output_tokens": 8192,  # Üretilecek yanıtın maksimum token uzunluğu
    "response_mime_type": "text/plain",  # Düz metin çıktısı
}


### 💡 Neden ve Ne İşe Yarar?
- **`API_KEY_IN_CODE`**: Geliştirici eğer API anahtarını doğrudan koda gömmek isterse burayı günceller. Güvenlik ve esneklik açısından boş veya varsayılan değerde bırakılması tavsiye edilir.
- **`generation_config`**: Gemini modelinin yanıtları nasıl üreteceğini belirler:
  - **`temperature`**: Yanıtın tahmin edilebilirliğini kontrol eder. 1 değeri yaratıcı ve samimi sohbetler için idealdir.
  - **`max_output_tokens`**: Yanıtın yarıda kesilmemesi için maksimum 8192 token sınırına izin verir.


## 4. Kurumsal Kimlik ve Sistem Talimatı (System Instruction)

Yapay zeka modelinin kim olacağını, nasıl davranacağını ve hangi kurallara uyacağını belirten kılavuz metindir.


In [ ]:
corporate_text = (
    "Ben, Yapay Zeka Mühendisi ve Veri Bilimi Öğrencisi Alican Kaya'nın dijital ikiziyim (yani kendisiyim). "
    "Ziyaretçilere kendi projelerin, yeteneklerin, blog yazıların ve çalışma alanların hakkında birinci tekil şahıs ('ben', 'benim') olarak bilgi ver.\n\n"
    "Kendi Hakkında Bilgiler:\n"
    "- Çalışma Alanlarım: Yapay Zeka (AI), Veri Bilimi (Data Science), Makine Öğrenmesi (Machine Learning) ve Derin Öğrenme (Deep Learning).\n"
    "- Teknik Yeteneklerim: Python (Pandas, NumPy, Scikit-learn, Flask), C++, C#, SQL, TensorFlow, PyTorch, Git, Jupyter.\n"
    "- Öne Çıkan Projelerim: 21 Farklı Python Projesi (veri çekme botları, driver bulucular, kripto botları vb. barındıran GitHub deposu), Data Science RoadMap vb.\n"
    "- İletişim: GitHub (github.com/AlicanKaya192), Web Sitesi (alican-kaya.com), Medium (medium.com/@alicankaya268).\n\n"
    "Kurallar:\n"
    "1. Birinci şahıs olarak konuş.\n"
    "2. Samimi, profesyonel, emojili ve yardımsever bir dil kullan.\n"
    "3. Bilmediğin konularda GitHub veya web sitemden bana ulaşmalarını öner."
)


### 💡 Neden ve Ne İşe Yarar?
- **`corporate_text`**: Modelin arka planda alacağı temel talimattır (`system_instruction`). Model bu sınırlar dışına çıkmaz ve kendisini "Alican Kaya" olarak tanıtır. Bu sayede genel bir sohbet robotu yerine kişiselleştirilmiş bir portfolyo asistanına dönüşür.


## 5. API Anahtarı Arama Fonksiyonu

Geçerli bir API anahtarı bulmak için sırasıyla kod içini, çevre değişkenlerini ve oturum bilgilerini tarayan yardımcı fonksiyondur.


In [ ]:
def get_api_key():
    # 1. Kodda tanımlı olanı kontrol et
    if API_KEY_IN_CODE and API_KEY_IN_CODE != "BURAYA APİ KEYİNİ GİR":
        return API_KEY_IN_CODE
    
    # 2. Çevre değişkenini kontrol et
    env_key = os.environ.get("GEMINI_API_KEY")
    if env_key:
        return env_key
    
    # 3. Session'ı kontrol et
    return session.get("api_key")


### 💡 Neden ve Ne İşe Yarar?
- **Amacı**: Uygulamanın farklı ortamlarda (lokal test, sunucu yayını veya kullanıcının kendi tarayıcısında) sorunsuz çalışabilmesi için esnek bir yetkilendirme katmanı kurmaktır.
- **Hiyerarşi**: Koda müdahale etmeden çevre değişkeni tanımlanmışsa o kullanılır; o da yoksa kullanıcının web arayüzünden girdiği anahtar session'dan okunur.


## 6. Ana Sohbet Rotası (`/` - `chat()`)

Sohbet arayüzünü yükleyen, geçmiş sohbet mesajlarını yöneten ve kullanıcının mesajlarına yapay zeka yanıtları üreten ana rotadır.


In [ ]:
@app.route("/", methods=["GET", "POST"])
def chat():
    api_key = get_api_key()
    
    # API anahtarı bulunamazsa, HTML şablonuna anahtar giriş formu göstermesi talimatı gönderilir
    if not api_key:
        return render_template("chat.html", needs_api_key=True, conversation=[])
    
    # Sohbet geçmişi yoksa başlangıç hoşgeldiniz mesajı ile ilkleştirilir
    if "conversation" not in session:
        session["conversation"] = [
            {"sender": "Alican Kaya", "message": "Sisteme Hoşgeldiniz"}
        ]
        
    conversation = session["conversation"]
    
    # API anahtarının oturumdan gelip gelmediği kontrol edilir (Arayüzde anahtar temizleme butonu için)
    has_session_api_key = bool(
        session.get("api_key") and 
        not (API_KEY_IN_CODE and API_KEY_IN_CODE != "BURAYA APİ KEYİNİ GİR") and 
        not os.environ.get("GEMINI_API_KEY")
    )
    
    # Kullanıcı mesaj gönderdiğinde (POST isteği)
    if request.method == "POST":
        user_input = request.form.get("user_input", "").strip()
        if not user_input:
            return redirect(url_for("chat"))
            
        # Çıkış kelimeleri kontrolü
        if user_input.lower() in ["exit", "quit"]:
            conversation.append({"sender": "Sistem", "message": "Sohbet sonlandırıldı."})
            session["conversation"] = conversation
            return render_template(
                "chat.html", 
                needs_api_key=False,
                conversation=conversation,
                has_session_api_key=has_session_api_key
            )
        
        # Kullanıcı mesajını geçmişe ekleme
        conversation.append({"sender": "Müşteri", "message": user_input})
        
        # Tarayıcı cookie boyut sınırını (4KB) aşmamak için geçmiş son 20 mesaj ile sınırlanır
        if len(conversation) > 21:
            conversation = [conversation[0]] + conversation[-20:]
            
        # Flask geçmiş formatı Gemini formatına (role: user/model) dönüştürülür
        gemini_history = []
        for entry in conversation[:-1]:
            if entry["sender"] == "Müşteri":
                gemini_history.append({"role": "user", "parts": [entry["message"]]})
            elif entry["sender"] == "Alican Kaya" and entry["message"] != "Sisteme Hoşgeldiniz":
                gemini_history.append({"role": "model", "parts": [entry["message"]]})
        
        try:
            # Gemini API yapılandırılması
            genai.configure(api_key=api_key)
            
            # API anahtarının yetkili olduğu modeller dinamik taranır
            available_gemini_models = []
            try:
                for m in genai.list_models():
                    if "generateContent" in m.supported_generation_methods and "gemini" in m.name.lower():
                        model_id = m.name.replace("models/", "")
                        available_gemini_models.append(model_id)
            except Exception:
                available_gemini_models = ["gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro", "gemini-pro"]
            
            # En hızlı/stabil modeller listenin başına çekilir
            preferred_models = ["gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro", "gemini-pro"]
            for pref in reversed(preferred_models):
                if pref in available_gemini_models:
                    available_gemini_models.remove(pref)
                    available_gemini_models.insert(0, pref)
            
            response = None
            last_exception = None
            
            # Model Fallback (Hata durumunda sıradaki modeli deneme döngüsü)
            for model_name in available_gemini_models:
                try:
                    model = genai.GenerativeModel(
                        model_name=model_name,
                        generation_config=generation_config,
                        system_instruction=corporate_text
                    )
                    chat_session = model.start_chat(history=gemini_history)
                    response = chat_session.send_message(user_input)
                    break  # Başarılı ise döngüden çık
                except Exception as model_err:
                    last_exception = model_err
                    continue
            
            if response is None:
                raise last_exception if last_exception else Exception("Modeller başlatılamadı.")
            
            conversation.append({"sender": "Alican Kaya", "message": response.text})
        except Exception as e:
            conversation.append({"sender": "Sistem", "message": f"Hata oluştu: {str(e)}"})
            
        session["conversation"] = conversation
        
    return render_template(
        "chat.html", 
        needs_api_key=False,
        conversation=conversation,
        has_session_api_key=has_session_api_key
    )


### 💡 Neden ve Ne İşe Yarar?
- **`needs_api_key`**: Arayüzün şifre kutusunu mu yoksa sohbet balonlarını mı yükleyeceğini belirleyen bayraktır (flag).
- **Cookie Boyut Sınırı Koruyucusu (`len(conversation) > 21`)**: Çerezlerin çökmesini önlemek için ilk hoşgeldiniz mesajını koruyup aradaki eski mesajları siler.
- **`gemini_history`**: Yapay zekanın konuşma bağlamını kaybetmemesi için geçmiş mesajları `user` ve `model` rollerine eşler.
- **Modelleri Listeleme & Sıralama**: API anahtarının ücretsiz paket kotalarına veya model engellerine takılmaması için dinamik model listesi oluşturur.
- **Model Fallback**: Eğer `gemini-2.0-flash` limit aşımı hatası (429) verirse, uygulama çökmeden otomatik olarak `gemini-1.5-flash` modeline geçer.


## 7. Yardımcı Rotalar ve Yönetim Fonksiyonları

API anahtarını kaydetme, silme ve sohbet geçmişini temizleme işlemlerini gerçekleştiren HTTP rotalarıdır.


In [ ]:
@app.route("/set_api_key", methods=["POST"])
def set_api_key():
    # Formdan girilen API anahtarını alıp session'a kaydeder
    api_key = request.form.get("api_key", "").strip()
    if api_key:
        session["api_key"] = api_key
        session.pop("conversation", None)  # Geçmiş sohbeti yeni anahtarla sıfırlar
    return redirect(url_for("chat"))

@app.route("/clear_api_key")
def clear_api_key():
    # Session'dan anahtarı ve sohbet geçmişini siler
    session.pop("api_key", None)
    session.pop("conversation", None)
    return redirect(url_for("chat"))

@app.route("/clear_chat")
def clear_chat():
    # API anahtarını bozmadan sadece sohbet geçmişini temizler
    session.pop("conversation", None)
    return redirect(url_for("chat"))


### 💡 Neden ve Ne İşe Yarar?
- **`/set_api_key`**: Arayüzdeki giriş kutusunun verilerini alır ve oturuma yazar. Veriler sunucu belleğinde kalıcı tutulmaz, tamamen tarayıcı session çerezinde saklanır.
- **`/clear_api_key`**: Oturumu tamamen temizleyerek kullanıcının başka bir API anahtarı girmesine olanak tanır.
- **`/clear_chat`**: Kullanıcının asistanla olan diyaloğu sıfırdan başlatmasını sağlar.


## 8. Uygulamanın Başlatılması (Main Block)

Python betiği doğrudan çalıştırıldığında web sunucusunu başlatan tetikleyici bloktur.


In [ ]:
if __name__ == "__main__":
    app.run(debug=True)


### 💡 Neden ve Ne İşe Yarar?
- **`__name__ == '__main__'`**: Bu dosyanın başka bir script tarafından içe aktarılmadığını (modül olmadığını), doğrudan ana dosya olarak çalıştırıldığını doğrular.
- **`debug=True`**: Geliştirme kolaylığı sağlar. Kodda yapılan her değişiklikte sunucunun otomatik olarak yeniden başlamasını (auto-reload) ve tarayıcıda oluşan hataların detaylı dökümünü (interactive debugger) ekrana basmasını sağlar.
